## Group-level fMRI analysis 

> Loads repeated fMRI responses across subjects and stories, computes group-level direct and indirect trajectory-curvature estimates across ROIs and temporal resolutions, and performs statistical analyses. The notebook also includes controls that equalize the number of voxels across ROIs and compare ROIs within matched SNR bins, and generates the group-average cortical map of implied curvature. This notebook supports Fig. 2b,c, Fig. 4, Fig. 6c, Extended Data Fig. 2, and Extended Data Fig. 4b,d.

In [ ]:
from pathlib import Path
import sys

# Find repository root
repo_root = Path.cwd()

while not (repo_root / "src").exists():
    if repo_root == repo_root.parent:
        raise RuntimeError("Could not find repository root containing src/")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


%matplotlib inline
%config InlineBackend.figure_format = "retina"

%reload_ext autoreload
%autoreload 2


import numpy as np
import cortex
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scipy.linalg
import statsmodels.api as sm
import statsmodels.formula.api as smf

from collections import defaultdict
from scipy.stats import wilcoxon

np.random.seed(0)


from src.fmri_core import (
    build_roi_voxs,
    equal_voxs,
    voxel_to_vertex,
    align_subj_to_fsaverage,
    load_response_trials,
    avg_pop_curvature,
    estimate_snr,
    compute_coherence_spectrum,
    lowpass_and_resample,
    get_xfm,
    zs,
    stories,
)

In [ ]:
def get_subj_data(data, story, subject, roi_order):
    y = np.full(len(roi_order), np.nan, dtype=float)

    for i, roi in enumerate(roi_order):
        val = data.get(story, {}).get(roi, {}).get(subject, np.nan)

        if np.isfinite(val):
            y[i] = val

    return y


def get_mean(data, stories, roi_order):
    mean = np.full(len(roi_order), np.nan)
    n = np.zeros(len(roi_order))

    for r, roi in enumerate(roi_order):
        all_vals = []

        for story in stories:
            subj2vals = data.get(story, {}).get(roi, {})
            vals = np.array(list(subj2vals.values()), dtype=float)
            vals = vals[np.isfinite(vals)]
            all_vals.extend(vals)

        if len(all_vals) > 0:
            mean[r] = float(np.mean(all_vals))
            n[r] = len(all_vals)

    return mean, n

def get_mean_for_fs(data, fs_out, stories, roi_order):
    """Compute the group mean for each ROI at one sampling rate."""
    means = np.full(len(roi_order), np.nan)

    for i, roi in enumerate(roi_order):
        values = []

        for story in stories:
            subject_values = (
                data
                .get(fs_out, {})
                .get(story, {})
                .get(roi, {})
            )

            values.extend(
                value
                for value in subject_values.values()
                if np.isfinite(value)
            )

        if values:
            means[i] = np.mean(values)

    return means

def make_main_df(curv_dict):
    rows = []

    for story in stories:
        for subject in subjects:
            for roi in roi_order:
                value = (
                    curv_dict
                    .get(story, {})
                    .get(roi, {})
                    .get(subject, np.nan)
                )

                if np.isfinite(value):
                    rows.append({
                        "curv": float(value),
                        "roi_idx": roi_to_idx[roi],
                        "subject": str(subject),
                        "story": str(story),
                    })

    return pd.DataFrame(rows)


def make_snr_df(curv_dict, bin_id):
    rows = []

    for story in stories:
        for subject in subjects:
            for roi in roi_order:
                value = (
                    curv_dict
                    .get(story, {})
                    .get(subject, {})
                    .get(roi, {})
                    .get(bin_id, np.nan)
                )

                if np.isfinite(value):
                    rows.append({
                        "curv": float(value),
                        "roi_idx": roi_to_idx[roi],
                        "subject": str(subject),
                        "story": str(story),
                    })

    return pd.DataFrame(rows)


def fit_mixed_model(df, label):
    model = smf.mixedlm(
        "curv ~ roi_idx",
        df,
        groups=df["subject"],
        vc_formula={"story": "0 + C(story)"}
    )

    fit = model.fit(
        reml=True,
        method="lbfgs"
    )

    beta = fit.params["roi_idx"]
    se = fit.bse["roi_idx"]
    p = fit.pvalues["roi_idx"]

    print(
        f"{label}: slope = {beta:.3f} ± {se:.3f} SE, "
        f"p = {p:.3g}"
    )

    return fit

#### Group-level ROI-averaged direct and indirect curvature analysis - original sampling (Fig 2a-b, Extended Data Fig 2)
Compute direct and AR(1)-based indirect curvature for every subject, story, and ROI using ROI-averaged responses. The analysis can be run with the original ROI voxel counts (Fig 2a-b) or with voxel counts equalized across ROIs as a control (Extended Data Figure 2b).

In [ ]:
# Analysis case:
# "original" = use all ROI voxels
# "equal_voxels" = equate voxel counts across ROIs
analysis_case = "original"

subjects = ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "BFD001", "BFD003"]

dir_roi_curv = defaultdict(lambda: defaultdict(dict))
ar1_roi_coef = defaultdict(lambda: defaultdict(dict))
ind_roi_curv = defaultdict(lambda: defaultdict(dict))

roi_voxs_subj = defaultdict(dict)
roi_voxs_subj_eq = defaultdict(dict)


# Build ROI voxel lists once per subject
for subject in subjects:
    roi_voxs = build_roi_voxs(subject)
    roi_voxs_eq = equal_voxs(roi_voxs)

    roi_voxs_subj[subject] = {
        roi: list(voxs) for roi, voxs in roi_voxs.items()
    }

    roi_voxs_subj_eq[subject] = {
        roi: list(voxs) for roi, voxs in roi_voxs_eq.items()
    }


# Select voxel sets
if analysis_case == "original":
    roi_voxels_by_subject = roi_voxs_subj
elif analysis_case == "equal_voxels":
    roi_voxels_by_subject = roi_voxs_subj_eq
else:
    raise ValueError("analysis_case must be 'original' or 'equal_voxels'")


# Main analysis
for story in stories:
    for subject in subjects:
        try:
            response_trials = load_response_trials(subject, story)
        except FileNotFoundError:
            print(f"Skipping {subject} - {story} (no data)")
            continue

        # Average across repeated runs
        run_avg_resp = np.mean(response_trials, axis=0)

        for roi, roi_voxels in roi_voxels_by_subject[subject].items():
            if len(roi_voxels) == 0:
                continue

            # Direct curvature
            curvature = avg_pop_curvature(run_avg_resp[:, roi_voxels].T)
            dir_roi_curv[story][roi][subject] = float(curvature)

            # AR(1) fit to ROI-averaged response
            ts = np.mean(run_avg_resp[:, roi_voxels], axis=1)
            ts_z = zs(ts)

            if not np.all(np.isfinite(ts_z)) or np.nanstd(ts_z) == 0:
                continue

            ar1_model = sm.tsa.AutoReg(ts_z, lags=1).fit()
            ar1_coef = float(ar1_model.params[1])
            ar1_roi_coef[story][roi][subject] = ar1_coef

            # Convert AR(1) coefficient to implied curvature
            theta = np.clip((ar1_coef - 1) / 2, -1, 1)
            ind_curvature = np.degrees(np.arccos(theta))
            ind_roi_curv[story][roi][subject] = float(ind_curvature)

In [ ]:
# Analysis case:
# "original" = use all ROI voxels
# "equal_voxels" = equate voxel counts across ROIs
analysis_case = "original"

dir_roi_curv = defaultdict(lambda: defaultdict(dict))
ar1_roi_coef = defaultdict(lambda: defaultdict(dict))
ind_roi_curv = defaultdict(lambda: defaultdict(dict))

roi_voxs_subj = defaultdict(dict)
roi_voxs_subj_eq = defaultdict(dict)

# Build ROI voxel lists once per subject
for subject in subjects:
    roi_voxs = build_roi_voxs(subject)
    roi_voxs_eq = equal_voxs(roi_voxs)

    roi_voxs_subj[subject] = {
        roi: list(voxs)
        for roi, voxs in roi_voxs.items()
    }

    roi_voxs_subj_eq[subject] = {
        roi: list(voxs)
        for roi, voxs in roi_voxs_eq.items()
    }


# Select voxel sets for this analysis
if analysis_case == "original":
    roi_voxels_by_subject = roi_voxs_subj

elif analysis_case == "equal_voxels":
    roi_voxels_by_subject = roi_voxs_subj_eq

else:
    raise ValueError(
        "analysis_case must be 'original' or 'equal_voxels'"
    )


# Main analysis loop
for story in stories:
    for subject in subjects:

        try:
            response_trials = load_response_trials(subject, story)
        except FileNotFoundError:
            print(f"Skipping {subject} - {story} (no data)")
            continue

        # Average responses across repeated runs
        run_avg_resp = np.mean(response_trials, axis=0)  # (time, voxels)

        roi_voxs = roi_voxels_by_subject[subject]

        for roi, roi_voxels in roi_voxs.items():

            if len(roi_voxels) == 0:
                continue

            # Direct curvature
            curvature = avg_pop_curvature(
                run_avg_resp[:, roi_voxels].T
            )

            dir_roi_curv[story][roi][subject] = float(curvature)

            # AR(1) fit to run-averaged, ROI-averaged response
            ts = np.mean(
                run_avg_resp[:, roi_voxels],
                axis=1
            )

            ts_z = zs(ts)

            if (
                not np.all(np.isfinite(ts_z))
                or np.nanstd(ts_z) == 0
            ):
                continue

            ar1_model = sm.tsa.AutoReg(
                ts_z,
                lags=1
            ).fit()

            ar1_coef = float(ar1_model.params[1])

            ar1_roi_coef[story][roi][subject] = ar1_coef

            # Convert AR(1) coefficient to implied curvature
            theta = (ar1_coef - 1) / 2
            theta = np.clip(theta, -1, 1)

            ind_curvature = np.degrees(
                np.arccos(theta)
            )

            ind_roi_curv[story][roi][subject] = float(ind_curvature)

In [ ]:
# ROI order used to access the data
roi_order = ["AC", "sPMv", "inferiorparietal", "prefrontal"]

# ROI labels 
roi_labels = ["AC", "sPMv", "AG", "prefrontal"]

fig, axes = plt.subplots(1, 2, figsize=(7, 5))
x = np.arange(len(roi_order))

# Assign one color to each subject
cmap = plt.cm.get_cmap("tab20", len(subjects))
subj_colors = {
    subj: cmap(i)
    for i, subj in enumerate(subjects)
}

legend_added = {
    subj: False
    for subj in subjects
}

# Direct curvature
ax = axes[0]
dir_mean, _ = get_mean(
    dir_roi_curv,
    stories,
    roi_order
)

for story in stories:
    for subject in subjects:
        y = get_subj_data(
            dir_roi_curv,
            story,
            subject,
            roi_order
        )

        if np.all(~np.isfinite(y)):
            continue

        label = subject if not legend_added[subject] else None

        ax.plot(
            x,
            y,
            linewidth=1,
            alpha=0.5,
            color=subj_colors[subject],
            label=label
        )

        legend_added[subject] = True

ax.plot(
    x,
    dir_mean,
    linewidth=2,
    marker="o",
    color="black",
    label="Group mean"
)

ax.set_title("Direct curvature")
ax.set_xticks(x)
ax.set_xticklabels(
    roi_labels,
    rotation=45,
    ha="right"
)
ax.set_ylabel("Curvature (degrees)")
ax.set_ylim(90, 120)


# Indirect curvature
ax = axes[1]
ind_mean, _ = get_mean(
    ind_roi_curv,
    stories,
    roi_order
)

for story in stories:
    for subject in subjects:
        y = get_subj_data(
            ind_roi_curv,
            story,
            subject,
            roi_order
        )

        if np.all(~np.isfinite(y)):
            continue

        ax.plot(
            x,
            y,
            linewidth=1,
            alpha=0.5,
            color=subj_colors[subject]
        )

ax.plot(
    x,
    ind_mean,
    linewidth=2,
    marker="o",
    color="black"
)

ax.set_title("Indirect curvature")
ax.set_xticks(x)
ax.set_xticklabels(
    roi_labels,
    rotation=45,
    ha="right"
)
ax.set_ylabel("Curvature (degrees)")
ax.set_ylim(90, 120)


# Legend
handles, labels = axes[0].get_legend_handles_labels()

leg = fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.98),
    ncol=4,
    frameon=True,
    fontsize=9,
    title="Subjects"
)

plt.tight_layout(rect=[0, 0, 1, 0.82])

# plt.savefig(
#     save_path,
#     format="pdf",
#     bbox_inches="tight",
#     bbox_extra_artists=(leg,)
# )

plt.show()

##### fMRI downsampling analysis (Fig 6c, Extended data fig 4b, d)
Resample run-averaged fMRI responses from the native 0.5 Hz sampling rate to slower temporal resolutions, then recompute direct and AR(1)-based indirect curvature across ROIs. This analysis tests whether hierarchical temporal straightening is preserved across temporal scales.

In [ ]:
# Fig. 6c / Extended Data Fig. 4b: low_pass = False
# Extended Data Fig. 4d: low_pass = True
low_pass = True

fs_outs = [0.5, 0.4, 0.3, 0.2]

roi_order = ["AC", "sPMv", "inferiorparietal", "prefrontal"]
roi_labels = ["AC", "sPMv", "AG", "prefrontal"]

# fs_out -> story -> roi -> subject -> value
dir_roi_curv_ds = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
ar1_roi_coef_ds = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
ind_roi_curv_ds = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

# Build ROI voxel lists once per subject
roi_voxs_subj = {}

for subject in subjects:
    roi_voxs = build_roi_voxs(subject)
    roi_voxs_subj[subject] = {
        roi: list(voxels) for roi, voxels in roi_voxs.items()
    }


# Compute direct and indirect curvature at each sampling rate
for story in stories:
    for subject in subjects:
        try:
            response_trials = load_response_trials(subject, story)
        except FileNotFoundError:
            print(f"Skipping {subject} - {story} (no data)")
            continue

        # Average across repeated runs
        run_avg_resp = np.mean(response_trials, axis=0)

        for fs_out in fs_outs:
            for roi, roi_voxels in roi_voxs_subj[subject].items():
                if len(roi_voxels) == 0:
                    continue

                # Downsample ROI responses
                resp = run_avg_resp[:, roi_voxels]

                resp_ds = lowpass_and_resample(
                    resp,
                    fs_in=0.5,
                    fs_out=fs_out,
                    low_pass=low_pass,
                )

                if resp_ds.shape[0] < 3:
                    continue

                # Direct curvature
                curvature = avg_pop_curvature(resp_ds.T)

                if np.isfinite(curvature):
                    dir_roi_curv_ds[fs_out][story][roi][subject] = float(curvature)

                # AR(1) fit to ROI-averaged response
                ts = np.mean(resp_ds, axis=1)
                ts_z = zs(ts)

                if not np.all(np.isfinite(ts_z)) or np.nanstd(ts_z) == 0:
                    continue

                ar1_model = sm.tsa.AutoReg(ts_z, lags=1, old_names=False).fit()
                ar1_coef = float(ar1_model.params[1])

                ar1_roi_coef_ds[fs_out][story][roi][subject] = ar1_coef

                # Convert AR(1) coefficient to implied curvature
                theta = np.clip((ar1_coef - 1) / 2, -1, 1)
                ind_curvature = np.degrees(np.arccos(theta))

                if np.isfinite(ind_curvature):
                    ind_roi_curv_ds[fs_out][story][roi][subject] = float(ind_curvature)

In [ ]:
# Plot group-average curvature across sampling rates
fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 5)
)

x = np.arange(len(roi_order))

colors = plt.cm.viridis(
    np.linspace(0, 1, len(fs_outs))
)

# Direct curvature
for color, fs_out in zip(colors, fs_outs):
    dir_mean = get_mean_for_fs(
        dir_roi_curv_ds,
        fs_out,
        stories,
        roi_order
    )

    axes[0].plot(
        x,
        dir_mean,
        marker="o",
        linewidth=2,
        color=color,
        label=f"{fs_out} Hz"
    )

# Indirect curvature
for color, fs_out in zip(colors, fs_outs):
    ind_mean = get_mean_for_fs(
        ind_roi_curv_ds,
        fs_out,
        stories,
        roi_order
    )

    axes[1].plot(
        x,
        ind_mean,
        marker="o",
        linewidth=2,
        color=color,
        label=f"{fs_out} Hz"
    )

# Format panels
for ax, title in zip(
    axes,
    ["Direct curvature", "Indirect curvature"]
):
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(
        roi_labels,
        rotation=45,
        ha="right"
    )
    ax.set_ylim(90, 120)
    ax.set_ylabel("Curvature (degrees)")
    ax.legend(title="Sampling rate")

plt.tight_layout()

# save_path = os.path.join(
#     fig_path,
#     "fmri_dir_ind_curv_group_downsample.pdf"
# )
# plt.savefig(
#     save_path,
#     format="pdf",
#     bbox_inches="tight"
# )

plt.show()

#### Statistics


#####  1) Non-parametric Wilcoxon test on direct and indirect curvatures:  
Compute the PFC–AC curvature difference for each subject–story dataset and test whether the pooled differences differ from zero. Median differences are reported as effect sizes.

In [ ]:
# Pooled subject × story differences
dir_deltas_pooled = []
ind_deltas_pooled = []

for story in stories:
    for subject in subjects:
        y_dir = get_subj_data(
            dir_roi_curv,
            story,
            subject,
            roi_order
        )

        delta_dir = y_dir[-1] - y_dir[0]

        if np.isfinite(delta_dir):
            dir_deltas_pooled.append(delta_dir)

        y_ind = get_subj_data(
            ind_roi_curv,
            story,
            subject,
            roi_order
        )

        delta_ind = y_ind[-1] - y_ind[0]

        if np.isfinite(delta_ind):
            ind_deltas_pooled.append(delta_ind)


dir_deltas_pooled = np.asarray(dir_deltas_pooled)
ind_deltas_pooled = np.asarray(ind_deltas_pooled)


# Wilcoxon signed-rank tests against zero
stat_dir, p_dir = wilcoxon(dir_deltas_pooled)
stat_ind, p_ind = wilcoxon(ind_deltas_pooled)

print("Direct curvature: W =", stat_dir, "p =", p_dir)
print("Indirect curvature: W =", stat_ind, "p =", p_ind)


# Median curvature difference: PFC - AC
median_dir = np.median(dir_deltas_pooled)
median_ind = np.median(ind_deltas_pooled)

print("Median direct difference:", median_dir)
print("Median indirect difference:", median_ind)

##### 2) Fit linear mixed-effect model to get slope

In [ ]:
roi_order = ["AC", "sPMv", "inferiorparietal", "prefrontal"]
roi_to_idx = {roi: i for i, roi in enumerate(roi_order)}

dir_df = make_main_df(dir_roi_curv)
ind_df = make_main_df(ind_roi_curv)

fit_dir = fit_mixed_model(
    dir_df,
    "Direct"
)

fit_ind = fit_mixed_model(
    ind_df,
    "Indirect"
)

#### SNR-matched control analysis (Fig 2c)
Compute indirect curvature separately for voxels grouped into SNR bins, then test the curvature trend across ROIs within each SNR bin using the same mixed-effects model as the main analysis.

In [ ]:
### SNR-matched control analysis

# Group voxels into matched SNR bins across ROIs and test whether
# indirect curvature still decreases across the cortical hierarchy.

snr_bin_edges = np.array([0.0, 0.2, 0.4, 0.6])
n_bins = len(snr_bin_edges) - 1

# Voxels with SNR > 0.6 were extremely sparse and absent from some ROIs,
# so this range was not included in the matched-SNR analysis.

# Compute voxelwise SNRs
roi_snrs = defaultdict(lambda: defaultdict(dict))

for story in stories:
    for subject in subjects:
        try:
            response_trials = load_response_trials(subject, story)
        except FileNotFoundError:
            print(f"Skipping {subject} - {story} (no data)")
            continue

        for roi, roi_voxels in roi_voxs_subj[subject].items():
            snrs = [
                estimate_snr(response_trials[:, :, voxel])
                for voxel in roi_voxels
            ]

            roi_snrs[story][subject][roi] = snrs


# Compute indirect curvature separately within each SNR bin
ind_roi_bin_curv = defaultdict(
    lambda: defaultdict(lambda: defaultdict(dict))
)

for story in stories:
    for subject in subjects:
        try:
            response_trials = load_response_trials(subject, story)
        except FileNotFoundError:
            continue

        roi_voxs = roi_voxs_subj[subject]

        for roi, snrs in roi_snrs[story][subject].items():
            snrs = np.asarray(snrs, dtype=float)

            bin_ids = np.digitize(
                snrs,
                snr_bin_edges,
                right=False
            )

            for bin_id in range(1, n_bins + 1):
                bin_voxels = [
                    voxel
                    for voxel, b in zip(roi_voxs[roi], bin_ids)
                    if b == bin_id
                ]

                if len(bin_voxels) == 0:
                    continue

                # Average responses across repetitions and voxels
                ts = np.mean(
                    response_trials[:, :, bin_voxels],
                    axis=(0, 2)
                )

                ts_z = zs(ts)

                if (
                    not np.all(np.isfinite(ts_z))
                    or np.nanstd(ts_z) == 0
                ):
                    continue

                # Fit AR(1)
                ar1_model = sm.tsa.AutoReg(
                    ts_z,
                    lags=1,
                    old_names=False
                ).fit()

                ar1_coef = float(
                    ar1_model.params[1]
                )

                # Convert AR(1) coefficient to implied curvature
                theta = np.clip(
                    (ar1_coef - 1) / 2,
                    -1,
                    1
                )

                curvature = np.degrees(
                    np.arccos(theta)
                )

                ind_roi_bin_curv[story][subject][roi][bin_id] = float(
                    curvature
                )

In [ ]:
# Fit the ROI trend separately within each SNR bin
for bin_id in range(1, n_bins + 1):
    snr_df = make_snr_df(
        ind_roi_bin_curv,
        bin_id
    )

    lower = snr_bin_edges[bin_id - 1]
    upper = snr_bin_edges[bin_id]

    fit_mixed_model(
        snr_df,
        f"SNR {lower:.1f}-{upper:.1f}"
    )

In [ ]:
# Mean indirect curvature across datasets for each SNR bin
x = np.arange(len(roi_order))
mean_by_bin = np.full((n_bins, len(roi_order)), np.nan)

for j, bin_id in enumerate(range(1, n_bins + 1)):
    all_y = []

    for story in stories:
        for subject in subjects:
            y = [
                ind_roi_bin_curv
                .get(story, {})
                .get(subject, {})
                .get(roi, {})
                .get(bin_id, np.nan)
                for roi in roi_order
            ]

            all_y.append(y)

    mean_by_bin[j] = np.nanmean(
        np.asarray(all_y, dtype=float),
        axis=0
    )


# Plot mean ROI curves for each SNR bin
fig, ax = plt.subplots(
    figsize=(4, 4),
    constrained_layout=True
)

for j in range(n_bins):
    label = (
        f"{snr_bin_edges[j]:.1f}–"
        f"{snr_bin_edges[j + 1]:.1f}"
    )

    ax.plot(
        x,
        mean_by_bin[j],
        marker="o",
        linewidth=2,
        label=label
    )

ax.set_xticks(x)
ax.set_xticklabels(
    roi_labels,
    rotation=45,
    ha="right"
)

ax.set_xlabel("ROI")
ax.set_ylabel("Indirect curvature (degrees)")
ax.set_ylim(90, 120)
ax.legend(
    title="SNR bin",
    frameon=False
)

plt.show()

# save_path = os.path.join(
#     fig_path,
#     "fmri_control_group.pdf"
# )
# plt.savefig(
#     save_path,
#     format="pdf",
#     bbox_inches="tight"
# )

#### Group-average cortical curvature map (Fig 4)
Estimate voxelwise AR(1) coefficients and coherence for each subject–story dataset, align maps to fsaverage, average across datasets, convert mean AR(1) coefficients to implied curvature, and display vertices passing the coherence threshold used in Fig. 4.

In [ ]:
sum_ar1 = None
count_ar1 = None
sum_coh = None
count_coh = None


# Estimate and accumulate AR(1) and coherence maps
for story in stories:
    for subject in subjects:
        try:
            response_trials = load_response_trials(subject, story)
        except FileNotFoundError:
            print(f"Skipping {subject} - {story} (no data)")
            continue

        # Run-averaged response
        run_avg_resp = np.mean(response_trials, axis=0)
        n_voxels = run_avg_resp.shape[1]

        # Voxelwise AR(1) coefficients
        ar1_vec = np.full(n_voxels, np.nan)

        for voxel in range(n_voxels):
            ts = run_avg_resp[:, voxel]

            if not np.all(np.isfinite(ts)):
                continue

            ts_z = zs(ts)

            if not np.all(np.isfinite(ts_z)) or np.nanstd(ts_z) == 0:
                continue

            ar1_model = sm.tsa.AutoReg(ts_z, lags=1, old_names=False).fit()
            ar1_vec[voxel] = ar1_model.params[1]

        # Align AR(1) map to fsaverage
        vert_ar1 = voxel_to_vertex(subject, ar1_vec)
        fs_ar1 = align_subj_to_fsaverage(subject, vert_ar1)

        # Voxelwise coherence
        _, _, _, _, _, coherence_lower = compute_coherence_spectrum(
            response_trials,
            fs=0.5,
            nfft=64
        )

        coh_vec = np.nanmean(coherence_lower, axis=0)

        vert_coh = voxel_to_vertex(subject, coh_vec)
        fs_coh = align_subj_to_fsaverage(subject, vert_coh)

        # Initialize fsaverage accumulators
        if sum_ar1 is None:
            n_vertices = len(fs_ar1)

            sum_ar1 = np.zeros(n_vertices, dtype=float)
            count_ar1 = np.zeros(n_vertices, dtype=int)

            sum_coh = np.zeros(n_vertices, dtype=float)
            count_coh = np.zeros(n_vertices, dtype=int)

        # Accumulate AR(1)
        valid_ar1 = np.isfinite(fs_ar1)
        sum_ar1[valid_ar1] += fs_ar1[valid_ar1]
        count_ar1[valid_ar1] += 1

        # Accumulate coherence
        valid_coh = np.isfinite(fs_coh)
        sum_coh[valid_coh] += fs_coh[valid_coh]
        count_coh[valid_coh] += 1


# Average across subject-story datasets
mean_ar1 = np.full_like(sum_ar1, np.nan, dtype=float)
valid_ar1 = count_ar1 > 0
mean_ar1[valid_ar1] = sum_ar1[valid_ar1] / count_ar1[valid_ar1]

mean_coh = np.full_like(sum_coh, np.nan, dtype=float)
valid_coh = count_coh > 0
mean_coh[valid_coh] = sum_coh[valid_coh] / count_coh[valid_coh]


# Convert mean AR(1) coefficients to implied curvature
theta = np.clip((mean_ar1 - 1) / 2, -1, 1)
ind_curv = np.degrees(np.arccos(theta))


# Apply coherence threshold
coh_percentile = 60
coh_floor = 0.05

coh_threshold = np.nanpercentile(mean_coh[valid_coh], coh_percentile)

keep_mask = (
    valid_coh
    & (mean_coh >= coh_threshold)
    & (mean_coh > coh_floor)
)

masked_curv = ind_curv.copy()
masked_curv[~keep_mask] = np.nan


# Plot group-average curvature map
kept_values = masked_curv[np.isfinite(masked_curv)]
vmin, vmax = np.nanpercentile(kept_values, [5, 95])

vertex = cortex.dataset.Vertex(
    masked_curv,
    subject="fsaverage",
    cmap="hot",
    vmin=vmin,
    vmax=vmax
)

flatmap_kwargs = dict(
    recache=False,
    pixelwise=True,
    sampler="nearest",
    height=700,
    with_rois=True,
    with_curvature=True,
    curvature_threshold=True,
    linewidth=4,
    nanmean=True
)

fig = cortex.quickshow(
    vertex,
    with_labels=False,
    colorbar_location=(0.9, 0.1, 0.4, 0.03),
    **flatmap_kwargs
)

cbar_ax = fig.axes[-1]
cbar_ax.set_xticks([vmin, vmax])
cbar_ax.set_xticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])

plt.show()